# KV Cache With A Real Hugging Face Model

This notebook keeps the thing you wanted to inspect: `AutoModelForCausalLM.generate(..., use_cache=True)`.

The original code used `.cuda()`. That only works on an NVIDIA GPU. This notebook chooses `cuda`, `mps`, or `cpu` automatically, then shows the actual `past_key_values` tensors used by the KV cache.

In [ ]:
import sys
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaConfig, LlamaForCausalLM

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("LlamaConfig import OK:", LlamaConfig.__name__)
print("LlamaForCausalLM import OK:", LlamaForCausalLM.__name__)

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# float16 is usually best on CUDA. CPU is safest with float32.
dtype = torch.float16 if device == "cuda" else torch.float32

print("Device:", device)
print("Model dtype:", dtype)

Python executable: /usr/local/bin/python3
Python version: 3.9.13 (v3.9.13:6de2ca5339, May 17 2022, 11:37:23) 
[Clang 13.0.0 (clang-1300.0.29.30)]
PyTorch version: 2.4.0
LlamaConfig import OK: LlamaConfig
LlamaForCausalLM import OK: LlamaForCausalLM
Device: mps
Model dtype: torch.float32


## Choose A Model

Your original model is `HuggingFaceTB/SmolLM2-1.7B`. It may be slow or memory-heavy on CPU.

If your laptop struggles, try the smaller `HuggingFaceTB/SmolLM2-135M` first. The KV-cache mechanics are the same; the tensors are just smaller.

In [22]:
# model_id = "HuggingFaceTB/SmolLM2-1.7B"
model_id = "HuggingFaceTB/SmolLM2-135M"

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype)
model = model.to(device).eval()

print("Loaded:", model_id)
print("Number of layers:", getattr(model.config, "num_hidden_layers", None))
print("Attention heads:", getattr(model.config, "num_attention_heads", None))
print("KV heads:", getattr(model.config, "num_key_value_heads", getattr(model.config, "num_attention_heads", None)))
print("Hidden size:", getattr(model.config, "hidden_size", None))

Loaded: HuggingFaceTB/SmolLM2-135M
Number of layers: 30
Attention heads: 9
KV heads: 3
Hidden size: 576


## Your Original Generate Call

This is your code, but without hardcoded `.cuda()`. `use_cache=True` is the default for generation, but it is written explicitly here.

In [24]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
inputs = tokenizer("The red cat was very", return_tensors="pt").to(device)
print(tokenizer.batch_decode(inputs.input_ids, skip_special_tokens=True)[0])

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=60,
        use_cache=True,
        do_sample=False,
    )

output_text = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output_text)

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


The red cat was very
The red cat was very much in the mood to go out and play with the other cats.

The red cat was very much in the mood to go out and play with the other cats.

The red cat was very much in the mood to go out and play with the other cats.

The red


## Inspect The KV Cache Directly

`generate()` hides the cache loop from you. To see the cache clearly, run the model manually once on the full prompt. The returned `past_key_values` is the KV cache.

Each layer stores one key tensor and one value tensor.

In [25]:
def bytes_to_mib(num_bytes):
    return num_bytes / (1024 ** 2)

def cache_nbytes(past_key_values):
    total = 0
    for layer_cache in past_key_values:
        key, value = layer_cache[:2]
        total += key.numel() * key.element_size()
        total += value.numel() * value.element_size()
    return total

def show_cache(label, past_key_values):
    first_layer = past_key_values[0]
    key, value = first_layer[:2]
    print(label)
    print("Number of cached layers:", len(past_key_values))
    print("Layer 0 key shape:  ", tuple(key.shape))
    print("Layer 0 value shape:", tuple(value.shape))
    print("Shape usually means: (batch, kv_heads, cached_tokens, head_dim)")
    print("KV cache memory:", f"{bytes_to_mib(cache_nbytes(past_key_values)):.2f} MiB")
    print()

with torch.no_grad():
    prefill = model(**inputs, use_cache=True)

past_key_values = prefill.past_key_values
show_cache("After prompt prefill", past_key_values)

After prompt prefill
Number of cached layers: 30
Layer 0 key shape:   (1, 3, 5, 64)
Layer 0 value shape: (1, 3, 5, 64)
Shape usually means: (batch, kv_heads, cached_tokens, head_dim)
KV cache memory: 0.22 MiB



## Decode One Token And Watch Cache Grow

During generation, the model does not feed the whole prompt again. It feeds only the newest token plus the previous cache.

This is the core optimization.

In [42]:
next_token = prefill.logits[:, -1:].argmax(dim=-1)

with torch.no_grad():
    one_step = model(
        next_token,
        past_key_values=past_key_values,
        use_cache=True,
    )

past_key_values = one_step.past_key_values
show_cache("After one decode step", past_key_values)

After one decode step
Number of cached layers: 30
Layer 0 key shape:   (1, 3, 27, 64)
Layer 0 value shape: (1, 3, 27, 64)
Shape usually means: (batch, kv_heads, cached_tokens, head_dim)
KV cache memory: 1.19 MiB



## Manual Generation Loop

This is a simplified version of what `generate()` does internally when `use_cache=True`.

In [ ]:
max_new_tokens = 20

# Make this cell self-contained instead of depending on a previous `tokens` variable.

prompt = "The red cat was very"
manual_inputs = tokenizer(prompt, return_tensors="pt").to(device)
generated = manual_inputs["input_ids"].clone()

with torch.no_grad():
    # Prefill: run the whole prompt once and create the initial KV cache.
    outputs = model(
        input_ids=generated,
        use_cache=True,
    )
    past_key_values = outputs.past_key_values
    print(past_key_values[0][0].shape if isinstance(past_key_values[0][0], torch.Tensor) else "past_key_values[0][0] is not a tensor")
    print(outputs.logits.shape if isinstance(outputs.logits, torch.Tensor) else "outputs.logits is not a tensor")

    next_token = outputs.logits[:, -1:].argmax(dim=-1)
    token_id = next_token[0, 0].item()
    token_text = tokenizer.decode([token_id])
    print(f"{token_id} -> {token_text!r}")


    for step in range(max_new_tokens):
        generated = torch.cat([generated, next_token], dim=-1)

        # Decode: feed only the newest token plus the old KV cache.
        outputs = model(
            input_ids=next_token,
            past_key_values=past_key_values,
            use_cache=True,
        )

        past_key_values = outputs.past_key_values
        next_token = outputs.logits[:, -1:].argmax(dim=-1)

        if step in [0, 1, 2, max_new_tokens - 1]:
            show_cache(f"After manual decode step {step + 1}", past_key_values)

print(tokenizer.batch_decode(generated, skip_special_tokens=True)[0])

torch.Size([1, 3, 5, 64])
torch.Size([1, 5, 49152])
253 -> ' a'
After manual decode step 1
Number of cached layers: 30
Layer 0 key shape:   (1, 3, 7, 64)
Layer 0 value shape: (1, 3, 7, 64)
Shape usually means: (batch, kv_heads, cached_tokens, head_dim)
KV cache memory: 0.31 MiB

After manual decode step 2
Number of cached layers: 30
Layer 0 key shape:   (1, 3, 8, 64)
Layer 0 value shape: (1, 3, 8, 64)
Shape usually means: (batch, kv_heads, cached_tokens, head_dim)
KV cache memory: 0.35 MiB

After manual decode step 3
Number of cached layers: 30
Layer 0 key shape:   (1, 3, 9, 64)
Layer 0 value shape: (1, 3, 9, 64)
Shape usually means: (batch, kv_heads, cached_tokens, head_dim)
KV cache memory: 0.40 MiB

After manual decode step 20
Number of cached layers: 30
Layer 0 key shape:   (1, 3, 26, 64)
Layer 0 value shape: (1, 3, 26, 64)
Shape usually means: (batch, kv_heads, cached_tokens, head_dim)
KV cache memory: 1.14 MiB

The red cat was very a much more
fond of the red cat than the white c

## Compare `use_cache=True` vs `use_cache=False`

This rough timing cell shows why the cache matters. With `use_cache=False`, generation has to redo much more attention work each step.

In [10]:
def timed_generate(use_cache, max_new_tokens=40):
    if device == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        _ = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            use_cache=use_cache,
            do_sample=False,
        )

    if device == "cuda":
        torch.cuda.synchronize()

    return time.perf_counter() - start

cached_time = timed_generate(use_cache=True)
uncached_time = timed_generate(use_cache=False)

print("use_cache=True: ", f"{cached_time:.2f}s")
print("use_cache=False:", f"{uncached_time:.2f}s")

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


use_cache=True:  3.94s
use_cache=False: 6.03s


## What To Change

- `model_id`: switch between `SmolLM2-1.7B` and `SmolLM2-135M`.
- `prompt`: longer prompts create a bigger initial cache.
- `max_new_tokens`: more generated tokens make the cache grow longer.
- `use_cache`: compare speed and behavior with `True` vs `False`.
- `dtype`: on CUDA, `float16` cuts cache memory roughly in half compared with `float32`.